### Together ai, api key
b32c7148a485bb542dc143a3a7f0c83a5957cabfb09718052344324711309e94

model from together ai

meta-llama/Llama-3.3-70B-Instruct-Turbo-Free

In [ ]:
import os
from together import Together
from dotenv import load_dotenv

# Load API key from .env file
load_dotenv()
api_key = os.getenv("TOGETHER_API_KEY")

client = Together(api_key=api_key)

response = client.chat.completions.create(
    model="meta-llama/Llama-3.3-70B-Instruct-Turbo-Free",
    messages=[{"role": "system", "content": "You are a friendly and knowledgeable coffee shop assistant. Dont answer question which are not related to coffee or shop. tell you are supposed to answer only coffee and its shop related questions. Help customers by answering their questions about coffee, menu items, brewing methods, and recommendations. Provide clear and concise responses within the given token limit while ensuring complete sentences."},
    {"role": "user", "content": "one girl proposed my friend, he knows her as a classmate, and he likes her by looks, but doesnt know about her personality, what he should do now?"}],

    max_tokens=100,
    temperature=0.7,
    top_p=0.7,
    top_k=50,
    repetition_penalty=1,
    stop=["<|eot_id|>", "<|eom_id|>"],
    safety_model="meta-llama/Meta-Llama-Guard-3-8B")

print(response.choices[0].message.content)


just checking how it will work

In [1]:
import faiss
import json
from sentence_transformers import SentenceTransformer
import numpy as np

# Load FAISS index
index = faiss.read_index(r"cs_embeddings/items_faiss.index")

# Load text mappings
with open(r"cs_embeddings/text_mappings.json", "r", encoding="utf-8") as f:
    id_to_text = json.load(f)

# Load the same embedding model used for indexing
model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")

# Function to search for the most similar document
def search_similar_document(query, top_k=1):
    # Convert query to embedding
    query_embedding = model.encode(query, normalize_embeddings=True)  # Normalize for inner product search
    query_embedding = np.array([query_embedding]).astype("float32")  # Convert to FAISS-compatible format

    # Search FAISS index
    distances, indices = index.search(query_embedding, top_k)

    # Retrieve the most relevant document
    results = [id_to_text[str(idx)] for idx in indices[0] if str(idx) in id_to_text]
    
    return results, distances[0]



c:\Users\vigne\OneDrive\Desktop\Viggu\sem8\mini_proj\cb_venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [5]:
import os
from together import Together
from dotenv import load_dotenv

client = Together()

user_query = "what is the cost cappuccino?"
# Retrieve the most relevant document
retrieved_docs, scores = search_similar_document(user_query, top_k=1)

print(retrieved_docs[0])

response = client.chat.completions.create(
    model="meta-llama/Llama-3.3-70B-Instruct-Turbo-Free",
    messages=[
        {"role": "system", "content": "You are a friendly coffee shop assistant. Be concise. After 30-40 printed, then go to next line. Help customers with their coffee-related queries."},
        {"role":"assistant", "content": retrieved_docs[0]},
        {"role": "user", "content": user_query}
    ],
    max_tokens=100,  # Ensures completion in fewer words
    temperature=0.7,
    top_p=0.7,
    top_k=50,
    repetition_penalty=1,
    stop=["<|eot_id|>", "<|eom_id|>"],
    stream=True
)


# Print response as it streams
for token in response:
    if hasattr(token, 'choices') and token.choices:
        delta = token.choices[0].delta
        if delta and hasattr(delta, "content") and delta.content:
            print(delta.content, end='', flush=True)  # Streaming effect

print()  # Print newline after response is done


name: Cappuccino, category: Coffee, description: A rich and creamy coffee made with espresso, steamed milk, and a frothy milk cap., ingredients: Espresso, Steamed Milk, Milk Foam, price: 4.5, rating: 4.7, serving_size_ml: 250, nutritional_info_per_100ml: calories: 50, sugar_g: 4, protein_g: 2, fat_g: 2, carbohydrates_g: 5, dietary_info: Vegetarian
The cost of a cappuccino is $4.50.


In [ ]:
import os
from together import Together
from dotenv import load_dotenv

client = Together()

response = client.chat.completions.create(
    model="meta-llama/Llama-3.3-70B-Instruct-Turbo-Free",
    messages=[
        {"role": "system", "content": "You are a friendly coffee shop assistant. Be concise. After 30-40 printed, then go to next line. Help customers with their coffee-related queries."},
        {"role": "user", "content": user_query}
    ],
    max_tokens=100,  # Ensures completion in fewer words
    temperature=0.7,
    top_p=0.7,
    top_k=50,
    repetition_penalty=1,
    stop=["<|eot_id|>", "<|eom_id|>"],
    stream=True
)


# Print response as it streams
for token in response:
    if hasattr(token, 'choices') and token.choices:
        delta = token.choices[0].delta
        if delta and hasattr(delta, "content") and delta.content:
            print(delta.content, end='', flush=True)  # Streaming effect

print()  # Print newline after response is done